In [1]:
import pandas as pd
from pathlib import Path
import logging
from sqlalchemy import create_engine, URL,text

In [2]:

connection_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password="Shantanu@2026",
    host="localhost",
    port=3306,
    database="test_cases"
)
engine = create_engine(connection_url)

In [3]:
p = Path(r"F:\00- MY JVM DOCS\Use Casese\Code\datasets-df-series\nba.csv")

In [4]:
df_sql = pd.read_sql("select * from nba_players",engine)

In [5]:
df_sql

,NAME,TEAM,NUMBER,POSITION,AGE,WEIGHT,COLLEGE,SALARY
0,Avery Bradley,Boston Celtics,0,PG,25,180,Texas,7730337.00
1,Jae Crowder,Boston Celtics,99,SF,25,235,Marquette,6796117.00
2,John Holland,Boston Celtics,30,SG,27,205,Boston University,4842684.11
3,R.J. Hunter,Boston Celtics,28,SG,22,185,Georgia State,1148640.00
4,Jonas Jerebko,Boston Celtics,8,PF,29,231,Unknown,5000000.00
...,...,...,...,...,...,...,...,...
452,Trey Lyles,Utah Jazz,41,PF,20,234,Kentucky,2239800.00
453,Shelvin Mack,Utah Jazz,8,PG,26,203,Butler,2433333.00
454,Raul Neto,Utah Jazz,25,PG,24,179,Unknown,900000.00
455,Tibor Pleiss,Utah Jazz,21,C,26,256,Unknown,2900000.00


In [6]:
logging.basicConfig(level=logging.INFO,force=True)

In [7]:
logging.debug("program is debuged")
logging.info("program is STARTED")
logging.warning("program is failing")
logging.error("program is failed")
logging.critical("program is in  critical stage")

INFO:root:program is STARTED
ERROR:root:program is failed
CRITICAL:root:program is in  critical stage


In [8]:
def read_data(p):
    try:
        logging.info("Loading Data")
        df = pd.read_csv(p)
        logging.info("Data Loaded Successfully")
        return df
    except Exception as e:
        logging.error(f"Error : {e} ")
        raise

In [9]:
df=read_data(p)

INFO:root:Loading Data
INFO:root:Data Loaded Successfully


In [10]:
print(df.info())

df.duplicated().sum()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 457 entries, 0 to 456
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      457 non-null    str    
 1   Team      457 non-null    str    
 2   Number    457 non-null    int64  
 3   Position  457 non-null    str    
 4   Age       457 non-null    int64  
 5   Height    457 non-null    str    
 6   Weight    457 non-null    int64  
 7   College   373 non-null    str    
 8   Salary    446 non-null    float64
dtypes: float64(1), int64(3), str(5)
memory usage: 32.3 KB
None


Name         0
Team         0
Number       0
Position     0
Age          0
Height       0
Weight       0
College     84
Salary      11
dtype: int64

In [ ]:
def tranformation(df):
    transforemed_df=df.drop_duplicates()
    transforemed_df.columns = transforemed_df.columns.str.upper()
    transforemed_df['COLLEGE']=transforemed_df['COLLEGE'].fillna('Unknown')
    transforemed_df['SALARY']=transforemed_df['SALARY'].fillna(transforemed_df['SALARY'].mean())
    transforemed_df = transforemed_df.drop('HEIGHT',axis=1)
    return transforemed_df

In [12]:
print(tranformation(df))

              NAME            TEAM  NUMBER POSITION  AGE  WEIGHT  \
0    Avery Bradley  Boston Celtics       0       PG   25     180   
1      Jae Crowder  Boston Celtics      99       SF   25     235   
2     John Holland  Boston Celtics      30       SG   27     205   
3      R.J. Hunter  Boston Celtics      28       SG   22     185   
4    Jonas Jerebko  Boston Celtics       8       PF   29     231   
..             ...             ...     ...      ...  ...     ...   
452     Trey Lyles       Utah Jazz      41       PF   20     234   
453   Shelvin Mack       Utah Jazz       8       PG   26     203   
454      Raul Neto       Utah Jazz      25       PG   24     179   
455   Tibor Pleiss       Utah Jazz      21        C   26     256   
456    Jeff Withey       Utah Jazz      24        C   26     231   

               COLLEGE        SALARY  
0                Texas  7.730337e+06  
1            Marquette  6.796117e+06  
2    Boston University  4.842684e+06  
3        Georgia State  1.1

In [13]:
def laod_data(df,table_name,engine):
    try:
        logging.info("Starting the Laoding")
        df.to_sql(name=table_name,
              con=engine,
              if_exists='append',
              index=False
              )
        logging.info("Laoding completed successfully")
    except Exception as e:
        logging.error(e)


In [14]:

def main_function():
    df = read_data(p)
    new_df = tranformation(df)
    laod_data(new_df,'nba_players',engine)
    

In [15]:
# import schedule
# import time

# def daily_task():
#     print("Executing daily ETL pipeline...")

# schedule.every(1).minute.do(daily_task)

# while True:
#     schedule.run_pending()
#     time.sleep(60)

In [31]:

def upsert_load(df, engine):
    with engine.begin() as conn:
        for _, row in df.iterrows():

            result = conn.execute(
                text("""
                    SELECT COUNT(*)
                    FROM nba_players
                    WHERE NAME = :name
                """),
                {"name": row["NAME"]}
            ).scalar()

            if result:
                conn.execute(
                    text("""
                        UPDATE nba_players
                        SET
                            TEAM = :team,
                            NUMBER = :number,
                            POSITION = :position,
                            AGE = :age,
                            WEIGHT = :weight,
                            COLLEGE = :college,
                            SALARY = :salary
                        WHERE NAME = :name
                    """),
                    {
                        "team": row["TEAM"],
                        "number": row["NUMBER"],
                        "position": row["POSITION"],
                        "age": row["AGE"],
                        "weight": row["WEIGHT"],
                        "college": row["COLLEGE"],
                        "salary": row["SALARY"],
                        "name": row["NAME"]
                    }
                )

            else:
                conn.execute(
                    text("""
                        INSERT INTO nba_players
                        (NAME, TEAM, NUMBER, POSITION, AGE, WEIGHT, COLLEGE, SALARY)
                        VALUES (:name, :team, :number, :position, :age, :weight, :college, :salary)
                    """),
                    {
                        "name": row["NAME"],
                        "team": row["TEAM"],
                        "number": row["NUMBER"],
                        "position": row["POSITION"],
                        "age": row["AGE"],
                        "weight": row["WEIGHT"],
                        "college": row["COLLEGE"],
                        "salary": row["SALARY"]
                    }
                )


In [19]:
df

,Name,Team,Number,Position,Age,Height,Weight,College,Salary
0,Avery Bradley,Boston Celtics,0,PG,25,06-Feb,180,Texas,7730337.0
1,Jae Crowder,Boston Celtics,99,SF,25,06-Jun,235,Marquette,6796117.0
2,John Holland,Boston Celtics,30,SG,27,06-May,205,Boston University,NaN
3,R.J. Hunter,Boston Celtics,28,SG,22,06-May,185,Georgia State,1148640.0
4,Jonas Jerebko,Boston Celtics,8,PF,29,06-Oct,231,NaN,5000000.0
...,...,...,...,...,...,...,...,...,...
452,Trey Lyles,Utah Jazz,41,PF,20,06-Oct,234,Kentucky,2239800.0
453,Shelvin Mack,Utah Jazz,8,PG,26,06-Mar,203,Butler,2433333.0
454,Raul Neto,Utah Jazz,25,PG,24,06-Jan,179,NaN,900000.0
455,Tibor Pleiss,Utah Jazz,21,C,26,07-Mar,256,NaN,2900000.0


In [20]:
df.loc[0,"Name"] ="Anderson silve"

In [21]:
df

,Name,Team,Number,Position,Age,Height,Weight,College,Salary
0,Anderson silve,Boston Celtics,0,PG,25,06-Feb,180,Texas,7730337.0
1,Jae Crowder,Boston Celtics,99,SF,25,06-Jun,235,Marquette,6796117.0
2,John Holland,Boston Celtics,30,SG,27,06-May,205,Boston University,NaN
3,R.J. Hunter,Boston Celtics,28,SG,22,06-May,185,Georgia State,1148640.0
4,Jonas Jerebko,Boston Celtics,8,PF,29,06-Oct,231,NaN,5000000.0
...,...,...,...,...,...,...,...,...,...
452,Trey Lyles,Utah Jazz,41,PF,20,06-Oct,234,Kentucky,2239800.0
453,Shelvin Mack,Utah Jazz,8,PG,26,06-Mar,203,Butler,2433333.0
454,Raul Neto,Utah Jazz,25,PG,24,06-Jan,179,NaN,900000.0
455,Tibor Pleiss,Utah Jazz,21,C,26,07-Mar,256,NaN,2900000.0


In [22]:
new_record = pd.DataFrame([{
    '': 456,
    'Name': 'Jeff Withey',
    'Team': 'Utah Jazz',
    'Number': 24,
    'Position': 'C',
    'Age': 26,
    'Height': '7-0',
    'Weight': 231,
    'College': 'Kansas',
    'Salary': 947276.0
}])

In [23]:
df = pd.concat([df, new_record], ignore_index=True)

In [24]:
df

,Name,Team,Number,Position,Age,Height,Weight,College,Salary,
0,Anderson silve,Boston Celtics,0,PG,25,06-Feb,180,Texas,7730337.0,NaN
1,Jae Crowder,Boston Celtics,99,SF,25,06-Jun,235,Marquette,6796117.0,NaN
2,John Holland,Boston Celtics,30,SG,27,06-May,205,Boston University,NaN,NaN
3,R.J. Hunter,Boston Celtics,28,SG,22,06-May,185,Georgia State,1148640.0,NaN
4,Jonas Jerebko,Boston Celtics,8,PF,29,06-Oct,231,NaN,5000000.0,NaN
...,...,...,...,...,...,...,...,...,...,...
453,Shelvin Mack,Utah Jazz,8,PG,26,06-Mar,203,Butler,2433333.0,NaN
454,Raul Neto,Utah Jazz,25,PG,24,06-Jan,179,NaN,900000.0,NaN
455,Tibor Pleiss,Utah Jazz,21,C,26,07-Mar,256,NaN,2900000.0,NaN
456,Jeff Withey,Utah Jazz,24,C,26,7-0,231,Kansas,947276.0,NaN


In [ ]:
new_df = tranformation(df)

In [27]:
new_df

,NAME,TEAM,NUMBER,POSITION,AGE,WEIGHT,COLLEGE,SALARY,
0,Anderson silve,Boston Celtics,0,PG,25,180,Texas,7.730337e+06,NaN
1,Jae Crowder,Boston Celtics,99,SF,25,235,Marquette,6.796117e+06,NaN
2,John Holland,Boston Celtics,30,SG,27,205,Boston University,4.833970e+06,NaN
3,R.J. Hunter,Boston Celtics,28,SG,22,185,Georgia State,1.148640e+06,NaN
4,Jonas Jerebko,Boston Celtics,8,PF,29,231,Unknown,5.000000e+06,NaN
...,...,...,...,...,...,...,...,...,...
453,Shelvin Mack,Utah Jazz,8,PG,26,203,Butler,2.433333e+06,NaN
454,Raul Neto,Utah Jazz,25,PG,24,179,Unknown,9.000000e+05,NaN
455,Tibor Pleiss,Utah Jazz,21,C,26,256,Unknown,2.900000e+06,NaN
456,Jeff Withey,Utah Jazz,24,C,26,231,Kansas,9.472760e+05,NaN


In [29]:
connection_url_2 = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password="Shantanu@2026",
    host="localhost",
    port=3306,
    database="test_cases"
)

engine2 = create_engine(connection_url_2)

In [32]:
upsert_load(new_df,engine)